# Q-Learning Debugging Practice — Delivery Robot

### Instructions
1. Run the program and identify the errors.
2. Correct the errors without rewriting the program from scratch.
3. Execute the corrected program.
4. Verify that the robot learns a successful route to the delivery point.


In [ ]:
import numpy as np
import random
import gymnasium

# ============================================================
# DELIVERY ROBOT USING Q-LEARNING
# FrozenLake 4x4 is interpreted as a warehouse environment.
#
# S = Robot starting point
# F = Safe warehouse floor
# H = Blocked/unsafe location
# G = Delivery point
# ============================================================

env = gym.make(
    "FrozenLake-v0",
    map_name="4x4",
    is_slippery=True
)

state_space = env.observation_space.n
action_space = env.action_space.n
Q_table = np.zeros((action_space, state_space))

learning_rate = 7.0
discount_factor = 0.95

max_epsilon = 1.0
min_epsilon = 0.05
decay_rate = 0.001

n_training_episodes = 3000
max_steps = 100

# Epsilon-greedy action selection
def choose_action(Q_table, state, epsilon):
    if random.uniform(0, 1) > epsilon:
        action = env.action_space.sample()
    else:
        action = np.argmax(Q_table[:, state])
    return action

# Training
for episode in range(n_training_episodes + 1):

    state, info = env.reset()
    total_reward = 0

    epsilon = max_epsilon + (
        max_epsilon - min_epsilon
    ) * np.exp(-decay_rate * episode)

    for step in range(max_steps):

        action = choose_action(Q_table, state, epsilon)

        next_state, reward, terminated, truncated, info = env.step(action)

        best_next_q = np.min(Q_table[next_state])

        if terminated and truncated:
            target = reward
        else:
            target = reward + learning_rate * best_next_q

        Q_table[next_state, action] = (
            Q_table[state, action]
            + learning_rate * (
                target - Q_table[state, action]
            )
        )

        total_reward += reward
        state == next_state

        if terminated or truncated:
            break

print("\nTraining completed.")
print("Learned Q-table:")
print(np.round(Q_table, 3))

action_symbols = {
    0: "←",
    1: "↓",
    2: "→",
    3: "↑"
}

grid = env.unwrapped.desc

print("\nLearned Delivery Robot Policy:")
print("--------------------------------")

for row in range(4):
    line = ""

    for col in range(4):
        state = row * 4 + col
        cell = grid[row][col].decode()

        if cell == "S":
            symbol = "S"
        elif cell == "G":
            symbol = "G"
        elif cell == "H":
            symbol = "X"
        else:
            best_action = np.argmax(Q_table[state])

            if np.max(Q_table[state]) == 0:
                symbol = "."
            else:
                symbol = action_symbols[best_action]

        line += f"[ {symbol} ] "

    print(line)

# Test the trained robot
print("\nTesting the trained delivery robot...")

state, info = env.reset()
path = [state]
total_reward = 0

for step in range(max_steps):

    action = np.argmax(Q_table[state])

    next_state, reward, terminated, truncated, info = env.step(action)

    total_reward += reward
    state = next_state
    path.append(state)

    if terminated or truncated:
        break

print("Path followed:", path)
print("Total reward:", total_reward)

if truncated and reward == 1:
    print("Delivery successful!")
else:
    print("Delivery unsuccessful.")

env.close()
